# UFC Fight Prediction — Model Training

Train a binary classifier on the prepared matchup data and build a prediction function
that takes two fighter names and predicts who would win.

**Input:** Prepared CSVs from `data/2026-03-18/prepared/` (generated by `fight_prediction_prep.ipynb`)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from pathlib import Path
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report, RocCurveDisplay

DATA_DIR = Path("../data/2026-03-18/prepared")
MODEL_DIR = Path("../models") / DATA_DIR.parent.name
MODEL_DIR.mkdir(parents=True, exist_ok=True)
sns.set_theme(style="whitegrid")

In [ ]:
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
fighter_features = pd.read_csv(DATA_DIR / "fighter_features.csv")
feature_cols = pd.read_csv(DATA_DIR / "feature_cols.csv", header=None)[0].tolist()

X_train, y_train = train[feature_cols], train["target"]
X_test, y_test = test[feature_cols], test["target"]

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Features: {len(feature_cols)}")
print(f"Target balance — Train: {y_train.mean():.3f}, Test: {y_test.mean():.3f}")

## Baseline Models

Compare three classifiers to find the best starting point:
1. **Logistic Regression** — simple linear baseline
2. **Random Forest** — ensemble of decision trees
3. **Gradient Boosting** — sequential boosted trees

In [ ]:
# Scale features for logistic regression
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=300, max_depth=10, random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=300, max_depth=4, learning_rate=0.05, random_state=42
    ),
}

results = {}
for name, model in models.items():
    # Use scaled data for logistic regression, raw for tree models
    X_tr = X_train_scaled if "Logistic" in name else X_train
    X_te = X_test_scaled if "Logistic" in name else X_test

    model.fit(X_tr, y_train)
    y_pred = model.predict(X_te)
    y_prob = model.predict_proba(X_te)[:, 1]

    acc = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)
    results[name] = {"accuracy": acc, "auc": auc, "model": model, "y_prob": y_prob}
    print(f"{name:25s}  Accuracy: {acc:.3f}  AUC: {auc:.3f}")

In [ ]:
# ROC curves for all models
fig, ax = plt.subplots(figsize=(8, 6))
for name, res in results.items():
    RocCurveDisplay.from_predictions(y_test, res["y_prob"], name=name, ax=ax)
ax.plot([0, 1], [0, 1], "k--", label="Random (AUC=0.5)")
ax.set_title("ROC Curves — Model Comparison")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Select the best model by AUC
best_name = max(results, key=lambda k: results[k]["auc"])
best_model = results[best_name]["model"]
print(f"Best model: {best_name} (AUC: {results[best_name]['auc']:.3f})")
print()
print(classification_report(
    y_test, best_model.predict(X_test_scaled if "Logistic" in best_name else X_test),
    target_names=["Fighter B wins", "Fighter A wins"]
))

In [ ]:
# Save the best model to disk
model_filename = best_name.lower().replace(" ", "_") + ".joblib"
joblib.dump(best_model, MODEL_DIR / model_filename)
print(f"Saved {best_name} to {MODEL_DIR / model_filename}")

## Feature Importance

In [ ]:
# Feature importance from the best tree-based model
if hasattr(best_model, "feature_importances_"):
    importances = best_model.feature_importances_
else:
    # For logistic regression, use absolute coefficients
    importances = np.abs(best_model.coef_[0])

feat_imp = pd.Series(importances, index=feature_cols).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 10))
feat_imp.tail(25).plot(kind="barh", ax=ax)
ax.set_title(f"Top 25 Feature Importances — {best_name}")
ax.set_xlabel("Importance")
plt.tight_layout()
plt.show()

## Done

The model has been saved to `models/2026-03-18/` with a name based on the winning algorithm (e.g. `gradient_boosting.joblib`). To make predictions, open `predict.ipynb`.